# Substation Digital Twin & Health Monitoring - Synthetic Data Generator

## Overview
This notebook generates synthetic data for substation digital twin monitoring:
- **Transformer DGA** - Dissolved gas analysis for oil-insulated transformers
- **Thermal monitoring** - Temperature sensors across equipment
- **Electrical measurements** - Voltage, current, power factor
- **Equipment health** - Calculated health indices

## Tables Created
| Table | Description | Volume |
|-------|-------------|--------|
| `dim_substations` | Substation registry | ~50 rows |
| `dim_equipment` | Transformers, breakers, etc. | ~500 rows |
| `fact_dga_analysis` | Dissolved gas samples | ~2K rows/month |
| `fact_thermal_readings` | Temperature telemetry | 200K+ rows/day |
| `fact_electrical_readings` | Power flow measurements | 300K+ rows/day |
| `fact_health_index` | Calculated health scores | ~5K rows/day |

## How to Run in Fabric
1. Attach this notebook to a Lakehouse
2. Run all cells sequentially
3. Data will be written as Delta tables

In [ ]:
!pip install faker

In [ ]:
# Configuration
SEED = 42
NUM_SUBSTATIONS = 50
EQUIPMENT_PER_SUBSTATION = 10

from datetime import datetime, timedelta
START_DATE = datetime(2024, 8, 1)  # Summer loading
END_DATE = datetime(2024, 8, 8)

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import uuid

np.random.seed(SEED)
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

print("Libraries loaded successfully")

## 1. Generate Substation Dimension

In [ ]:
def generate_substations(num_substations):
    substations = []
    
    voltage_classes = [
        ('Transmission', [345, 230, 161, 138]),
        ('Sub-Transmission', [69, 46, 34.5]),
        ('Distribution', [25, 15, 12.47, 4.16])
    ]
    
    for i in range(num_substations):
        sub_class, voltages = random.choice(voltage_classes)
        high_side = random.choice(voltages)
        low_side = random.choice([v for v in voltages if v < high_side] or [high_side / 3])
        
        substations.append({
            'substation_id': f'SUB-{str(i+1).zfill(3)}',
            'substation_name': f'{fake.city()} {sub_class} Substation',
            'substation_class': sub_class,
            'latitude': round(random.uniform(30, 48), 4),
            'longitude': round(random.uniform(-120, -75), 4),
            'high_side_kv': high_side,
            'low_side_kv': round(low_side, 2),
            'install_year': random.randint(1960, 2020),
            'rated_mva': random.choice([50, 100, 150, 200, 300, 500]),
            'configuration': random.choice(['Ring Bus', 'Breaker-and-a-Half', 'Double Bus']),
            'automation_level': random.choice(['Full SCADA', 'Partial SCADA', 'Manual']),
            'region': random.choice(['Northeast', 'Southeast', 'Midwest', 'West', 'Texas']),
            'criticality': random.choices(['Critical', 'High', 'Medium', 'Low'], weights=[0.15, 0.3, 0.4, 0.15])[0]
        })
    
    return pd.DataFrame(substations)

df_substations = generate_substations(NUM_SUBSTATIONS)
print(f"Generated {len(df_substations)} substations")
print(f"By class: {df_substations['substation_class'].value_counts().to_dict()}")
df_substations.head()

In [ ]:
def generate_equipment(substations_df, equipment_per_sub):
    equipment = []
    
    equipment_types = [
        ('Power Transformer', 0.3),
        ('Circuit Breaker', 0.25),
        ('Disconnect Switch', 0.15),
        ('Instrument Transformer', 0.15),
        ('Capacitor Bank', 0.08),
        ('Voltage Regulator', 0.07)
    ]
    
    manufacturers = ['ABB', 'Siemens', 'GE', 'Hitachi', 'Schneider', 'Eaton']
    
    for _, sub in substations_df.iterrows():
        for i in range(equipment_per_sub):
            eq_type = random.choices([e[0] for e in equipment_types], 
                                     weights=[e[1] for e in equipment_types])[0]
            
            install_year = max(sub['install_year'], random.randint(sub['install_year'], 2023))
            age = 2024 - install_year
            
            # Expected life varies by equipment type
            expected_life = {'Power Transformer': 40, 'Circuit Breaker': 35,
                            'Disconnect Switch': 50, 'Instrument Transformer': 35,
                            'Capacitor Bank': 25, 'Voltage Regulator': 30}[eq_type]
            
            equipment.append({
                'equipment_id': f'{sub["substation_id"]}-{eq_type[:3].upper()}-{str(i+1).zfill(2)}',
                'substation_id': sub['substation_id'],
                'equipment_type': eq_type,
                'manufacturer': random.choice(manufacturers),
                'model': f'{fake.lexify(text="????").upper()}-{random.randint(100, 999)}',
                'serial_number': fake.uuid4()[:12].upper(),
                'install_year': install_year,
                'age_years': age,
                'expected_life_years': expected_life,
                'life_consumed_pct': round(min(100, (age / expected_life) * 100), 1),
                'rated_voltage_kv': sub['high_side_kv'],
                'rated_current_a': random.choice([600, 1200, 2000, 3000]),
                'cooling_type': random.choice(['ONAN', 'ONAF', 'OFAF', 'Air']) if eq_type == 'Power Transformer' else 'N/A',
                'last_maintenance_date': fake.date_between(start_date='-2y', end_date='today'),
                'maintenance_status': random.choices(['Good', 'Monitor', 'Maintenance Due', 'Critical'],
                                                    weights=[0.6, 0.25, 0.12, 0.03])[0]
            })
    
    return pd.DataFrame(equipment)

df_equipment = generate_equipment(df_substations, EQUIPMENT_PER_SUBSTATION)
print(f"Generated {len(df_equipment)} equipment records")
print(f"By type: {df_equipment['equipment_type'].value_counts().to_dict()}")
df_equipment.head()

## 2. Generate DGA Analysis Data

In [ ]:
def calculate_duval_triangle(h2, ch4, c2h2, c2h4, c2h6):
    """Simplified Duval Triangle interpretation"""
    total = ch4 + c2h2 + c2h4
    if total == 0:
        return 'Normal'
    
    ch4_pct = ch4 / total * 100
    c2h2_pct = c2h2 / total * 100
    c2h4_pct = c2h4 / total * 100
    
    if c2h2_pct > 29:
        return 'High-Energy Discharge'
    elif c2h4_pct > 50:
        return 'Thermal Fault'
    elif ch4_pct > 98:
        return 'Partial Discharge'
    else:
        return 'Normal'

def generate_dga_analysis(equipment_df, samples_per_unit=4):
    """Generate dissolved gas analysis results"""
    
    # Filter to transformers only
    transformers = equipment_df[equipment_df['equipment_type'] == 'Power Transformer']
    
    samples = []
    for _, xfmr in transformers.iterrows():
        age_factor = xfmr['age_years'] / 40  # Older units have higher gas levels
        
        for i in range(samples_per_unit):
            sample_date = fake.date_between(start_date='-1y', end_date='today')
            
            # Base gas levels (ppm) - increase with age
            h2 = max(0, np.random.lognormal(2 + age_factor, 0.8))
            ch4 = max(0, np.random.lognormal(1.5 + age_factor * 0.5, 0.7))
            c2h6 = max(0, np.random.lognormal(1 + age_factor * 0.3, 0.6))
            c2h4 = max(0, np.random.lognormal(0.5 + age_factor * 0.4, 0.5))
            c2h2 = max(0, np.random.exponential(0.5 + age_factor * 0.3))  # Acetylene rare
            co = max(0, np.random.lognormal(3 + age_factor * 0.5, 0.8))
            co2 = max(0, np.random.lognormal(5 + age_factor * 0.3, 0.7))
            
            # Calculate diagnostic ratios
            fault_type = calculate_duval_triangle(h2, ch4, c2h2, c2h4, c2h6)
            
            # Total dissolved combustible gas
            tdcg = h2 + ch4 + c2h6 + c2h4 + c2h2 + co
            
            # Condition assessment
            if tdcg < 720:
                condition = 'Good'
            elif tdcg < 1920:
                condition = 'Monitor'
            elif tdcg < 4630:
                condition = 'Investigate'
            else:
                condition = 'Critical'
            
            samples.append({
                'sample_id': str(uuid.uuid4()),
                'equipment_id': xfmr['equipment_id'],
                'substation_id': xfmr['substation_id'],
                'sample_date': sample_date,
                'hydrogen_ppm': round(h2, 1),
                'methane_ppm': round(ch4, 1),
                'ethane_ppm': round(c2h6, 1),
                'ethylene_ppm': round(c2h4, 1),
                'acetylene_ppm': round(c2h2, 2),
                'carbon_monoxide_ppm': round(co, 1),
                'carbon_dioxide_ppm': round(co2, 1),
                'tdcg_ppm': round(tdcg, 1),
                'moisture_ppm': round(max(5, np.random.lognormal(2, 0.5)), 1),
                'oil_temperature_c': round(random.uniform(45, 75), 1),
                'duval_interpretation': fault_type,
                'condition_assessment': condition,
                'lab_name': random.choice(['TestLab Inc', 'OilAnalysis Corp', 'TechDiagnostics'])
            })
    
    return pd.DataFrame(samples)

df_dga = generate_dga_analysis(df_equipment)
print(f"Generated {len(df_dga)} DGA samples")
print(f"Condition: {df_dga['condition_assessment'].value_counts().to_dict()}")
print(f"Fault types: {df_dga['duval_interpretation'].value_counts().to_dict()}")
df_dga.head()

## 3. Generate Thermal Readings

In [ ]:
def generate_thermal_readings(equipment_df, substations_df, start_time, hours=24):
    """Generate thermal monitoring data"""
    readings = []
    
    for _, eq in equipment_df.iterrows():
        sub = substations_df[substations_df['substation_id'] == eq['substation_id']].iloc[0]
        
        # Base temperature depends on equipment type and loading
        base_temps = {
            'Power Transformer': 55,
            'Circuit Breaker': 35,
            'Disconnect Switch': 30,
            'Instrument Transformer': 40,
            'Capacitor Bank': 35,
            'Voltage Regulator': 45
        }
        base_temp = base_temps.get(eq['equipment_type'], 40)
        
        # Age increases base temperature
        age_factor = eq['age_years'] / 50 * 5
        
        current_time = start_time
        while current_time < start_time + timedelta(hours=hours):
            hour = current_time.hour
            
            # Ambient temperature pattern
            ambient = 25 + 10 * np.sin((hour - 6) * np.pi / 12) + np.random.normal(0, 2)
            
            # Load-dependent temperature rise
            load_factor = 0.6 + 0.3 * np.sin((hour - 8) * np.pi / 12)  # Peak at 2pm
            load_rise = 20 * load_factor
            
            equipment_temp = base_temp + age_factor + load_rise + np.random.normal(0, 3)
            
            # Hot spot is higher than winding average
            hot_spot = equipment_temp + random.uniform(5, 15) if eq['equipment_type'] == 'Power Transformer' else equipment_temp
            
            # Delta T from ambient
            delta_t = equipment_temp - ambient
            
            # Alarm conditions
            is_alarm = hot_spot > 85 or delta_t > 50
            
            readings.append({
                'reading_id': str(uuid.uuid4()),
                'equipment_id': eq['equipment_id'],
                'substation_id': eq['substation_id'],
                'timestamp': current_time,
                'ambient_temp_c': round(ambient, 1),
                'top_oil_temp_c': round(equipment_temp, 1) if eq['equipment_type'] == 'Power Transformer' else None,
                'winding_temp_c': round(equipment_temp + 5, 1) if eq['equipment_type'] == 'Power Transformer' else None,
                'hot_spot_temp_c': round(hot_spot, 1),
                'delta_t_c': round(delta_t, 1),
                'load_pct': round(load_factor * 100, 1),
                'cooling_stage': int(load_factor * 3) if eq['cooling_type'] in ['ONAF', 'OFAF'] else 0,
                'thermal_alarm': is_alarm
            })
            
            current_time += timedelta(minutes=15)
    
    return pd.DataFrame(readings)

# Generate for first 50 equipment items
df_thermal = generate_thermal_readings(df_equipment.head(50), df_substations, START_DATE, hours=24)
print(f"Generated {len(df_thermal)} thermal readings")
print(f"Thermal alarms: {df_thermal['thermal_alarm'].sum()}")
df_thermal.head()

## 4. Generate Electrical Readings

In [ ]:
def generate_electrical_readings(substations_df, start_time, hours=24):
    """Generate electrical measurements at substation level"""
    readings = []
    
    for _, sub in substations_df.iterrows():
        current_time = start_time
        
        while current_time < start_time + timedelta(hours=hours):
            hour = current_time.hour
            is_weekend = current_time.weekday() >= 5
            
            # Load pattern
            if is_weekend:
                base_load = 0.5
            else:
                if 8 <= hour <= 20:
                    base_load = 0.7 + 0.25 * np.sin((hour - 8) * np.pi / 12)
                else:
                    base_load = 0.4
            
            load_mva = sub['rated_mva'] * base_load * (1 + np.random.normal(0, 0.05))
            
            # Power factor (industrial loads have worse PF)
            pf = random.uniform(0.85, 0.98)
            
            # Active and reactive power
            mw = load_mva * pf
            mvar = load_mva * np.sqrt(1 - pf**2)
            
            # Voltage (per unit, nominal = 1.0)
            voltage_pu = 1.0 + np.random.normal(0, 0.02)
            voltage_pu = np.clip(voltage_pu, 0.95, 1.05)
            
            # Current
            current_a = load_mva * 1000 / (sub['high_side_kv'] * np.sqrt(3))
            
            # Frequency
            frequency = 60.0 + np.random.normal(0, 0.01)
            
            readings.append({
                'reading_id': str(uuid.uuid4()),
                'substation_id': sub['substation_id'],
                'timestamp': current_time,
                'voltage_kv': round(sub['high_side_kv'] * voltage_pu, 2),
                'voltage_pu': round(voltage_pu, 4),
                'current_a': round(current_a, 1),
                'active_power_mw': round(mw, 2),
                'reactive_power_mvar': round(mvar, 2),
                'apparent_power_mva': round(load_mva, 2),
                'power_factor': round(pf, 3),
                'frequency_hz': round(frequency, 3),
                'load_pct': round(load_mva / sub['rated_mva'] * 100, 1),
                'energy_mwh': round(mw / 4, 2)  # 15-minute energy
            })
            
            current_time += timedelta(minutes=15)
    
    return pd.DataFrame(readings)

df_electrical = generate_electrical_readings(df_substations, START_DATE, hours=24)
print(f"Generated {len(df_electrical)} electrical readings")
print(f"Load range: {df_electrical['load_pct'].min():.1f}% - {df_electrical['load_pct'].max():.1f}%")
df_electrical.head()

## 5. Generate Health Index Data

In [ ]:
def calculate_health_index(age, expected_life, dga_condition, thermal_status, maintenance_status):
    """Calculate equipment health index (0-100)"""
    
    # Age score (0-30 points)
    life_consumed = min(1.0, age / expected_life)
    age_score = 30 * (1 - life_consumed)
    
    # DGA score (0-25 points) - transformers only
    dga_scores = {'Good': 25, 'Monitor': 18, 'Investigate': 10, 'Critical': 0, None: 25}
    dga_score = dga_scores.get(dga_condition, 25)
    
    # Thermal score (0-25 points)
    thermal_scores = {'Normal': 25, 'Elevated': 15, 'High': 5, 'Critical': 0}
    thermal_score = thermal_scores.get(thermal_status, 25)
    
    # Maintenance score (0-20 points)
    maint_scores = {'Good': 20, 'Monitor': 15, 'Maintenance Due': 8, 'Critical': 0}
    maint_score = maint_scores.get(maintenance_status, 15)
    
    return age_score + dga_score + thermal_score + maint_score

def generate_health_index(equipment_df, dga_df, thermal_df, timestamp):
    """Generate health index scores for all equipment"""
    health_records = []
    
    for _, eq in equipment_df.iterrows():
        # Get latest DGA for transformers
        dga_condition = None
        if eq['equipment_type'] == 'Power Transformer':
            eq_dga = dga_df[dga_df['equipment_id'] == eq['equipment_id']]
            if len(eq_dga) > 0:
                dga_condition = eq_dga.sort_values('sample_date', ascending=False).iloc[0]['condition_assessment']
        
        # Get thermal status
        eq_thermal = thermal_df[thermal_df['equipment_id'] == eq['equipment_id']]
        if len(eq_thermal) > 0:
            max_temp = eq_thermal['hot_spot_temp_c'].max()
            if max_temp > 85:
                thermal_status = 'Critical'
            elif max_temp > 75:
                thermal_status = 'High'
            elif max_temp > 65:
                thermal_status = 'Elevated'
            else:
                thermal_status = 'Normal'
        else:
            thermal_status = 'Normal'
        
        # Calculate health index
        hi = calculate_health_index(
            eq['age_years'],
            eq['expected_life_years'],
            dga_condition,
            thermal_status,
            eq['maintenance_status']
        )
        
        # Risk category
        if hi >= 80:
            risk = 'Low'
        elif hi >= 60:
            risk = 'Moderate'
        elif hi >= 40:
            risk = 'High'
        else:
            risk = 'Critical'
        
        health_records.append({
            'record_id': str(uuid.uuid4()),
            'equipment_id': eq['equipment_id'],
            'substation_id': eq['substation_id'],
            'equipment_type': eq['equipment_type'],
            'timestamp': timestamp,
            'health_index': round(hi, 1),
            'risk_category': risk,
            'age_score': round(30 * (1 - min(1.0, eq['age_years'] / eq['expected_life_years'])), 1),
            'dga_condition': dga_condition,
            'thermal_status': thermal_status,
            'maintenance_status': eq['maintenance_status'],
            'recommended_action': 'Inspect' if hi < 60 else ('Monitor' if hi < 80 else 'None'),
            'estimated_remaining_life_years': round(eq['expected_life_years'] * (hi / 100), 0)
        })
    
    return pd.DataFrame(health_records)

df_health = generate_health_index(df_equipment, df_dga, df_thermal, START_DATE)
print(f"Generated {len(df_health)} health index records")
print(f"Risk breakdown: {df_health['risk_category'].value_counts().to_dict()}")
print(f"Avg health index: {df_health['health_index'].mean():.1f}")
df_health.head()

## 6. Visualize Substation Health

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Health Index Distribution
axes[0, 0].hist(df_health['health_index'], bins=20, color='steelblue', edgecolor='white')
axes[0, 0].axvline(x=60, color='orange', linestyle='--', label='High Risk Threshold')
axes[0, 0].axvline(x=40, color='red', linestyle='--', label='Critical Threshold')
axes[0, 0].set_title('Equipment Health Index Distribution')
axes[0, 0].set_xlabel('Health Index')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

# Health by Equipment Type
health_by_type = df_health.groupby('equipment_type')['health_index'].mean().sort_values()
health_by_type.plot(kind='barh', ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Average Health Index by Equipment Type')
axes[0, 1].set_xlabel('Health Index')

# DGA Condition Breakdown
dga_counts = df_dga['condition_assessment'].value_counts()
colors = {'Good': 'green', 'Monitor': 'yellow', 'Investigate': 'orange', 'Critical': 'red'}
dga_counts.plot(kind='pie', ax=axes[1, 0], colors=[colors.get(c, 'gray') for c in dga_counts.index],
                autopct='%1.0f%%')
axes[1, 0].set_title('DGA Condition Assessment')
axes[1, 0].set_ylabel('')

# Load Profile (first substation)
sub1_electrical = df_electrical[df_electrical['substation_id'] == df_substations.iloc[0]['substation_id']]
axes[1, 1].plot(sub1_electrical['timestamp'], sub1_electrical['load_pct'], color='blue', linewidth=1)
axes[1, 1].axhline(y=80, color='orange', linestyle='--', label='High Load')
axes[1, 1].axhline(y=100, color='red', linestyle='--', label='Rated')
axes[1, 1].set_title(f'Load Profile - {df_substations.iloc[0]["substation_name"]}')
axes[1, 1].set_ylabel('Load (%)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Data Validation

In [ ]:
print("=== Data Validation Report ===")
print(f"\nSubstations: {len(df_substations)} records")
print(f"  - By class: {df_substations['substation_class'].value_counts().to_dict()}")
print(f"  - Total capacity: {df_substations['rated_mva'].sum():,} MVA")

print(f"\nEquipment: {len(df_equipment)} records")
print(f"  - By type: {df_equipment['equipment_type'].value_counts().to_dict()}")
print(f"  - Valid FK: {df_equipment['substation_id'].isin(df_substations['substation_id']).all()}")

print(f"\nDGA Analysis: {len(df_dga)} records")
print(f"  - Conditions: {df_dga['condition_assessment'].value_counts().to_dict()}")

print(f"\nThermal Readings: {len(df_thermal)} records")
print(f"  - Hot spot range: {df_thermal['hot_spot_temp_c'].min():.1f}°C - {df_thermal['hot_spot_temp_c'].max():.1f}°C")

print(f"\nElectrical Readings: {len(df_electrical)} records")
print(f"  - Total energy: {df_electrical['energy_mwh'].sum():,.0f} MWh")

print(f"\nHealth Index: {len(df_health)} records")
print(f"  - Critical equipment: {len(df_health[df_health['risk_category'] == 'Critical'])}")
print(f"  - High risk equipment: {len(df_health[df_health['risk_category'] == 'High'])}")

## 8. Write to Lakehouse

In [ ]:
# Uncomment for Fabric execution
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()

# spark.createDataFrame(df_substations).write.mode("overwrite").format("delta").saveAsTable("dim_substations")
# spark.createDataFrame(df_equipment).write.mode("overwrite").format("delta").saveAsTable("dim_equipment")
# spark.createDataFrame(df_dga).write.mode("append").format("delta").saveAsTable("fact_dga_analysis")
# spark.createDataFrame(df_thermal).write.mode("append").format("delta").saveAsTable("fact_thermal_readings")
# spark.createDataFrame(df_electrical).write.mode("append").format("delta").saveAsTable("fact_electrical_readings")
# spark.createDataFrame(df_health).write.mode("append").format("delta").saveAsTable("fact_health_index")

# Local testing
df_substations.to_csv('dim_substations.csv', index=False)
df_health.to_csv('fact_health_index.csv', index=False)
print("Data saved successfully")

## Known Limitations
1. Health index formula is simplified - actual IEEE standards have more factors
2. DGA ratios don't include all diagnostic techniques (Rogers, Key Gas)
3. Equipment relationships (e.g., which breaker protects which transformer) not modeled

## Next Steps
1. Build Digital Twin model in Digital Twin Builder
2. Create real-time health index updates via Eventhouse
3. Configure Activator alerts for critical equipment conditions